# Notebook 07 — Front ByteTrack Buffer Ablation

**Objective:** compare ByteTrack buffers 15, 30, and 60 on the accepted one-fish Front video while holding the detector, thresholds, association parameters, video, and visibility ground truth fixed.

`OPERATIONAL_HIGH_CONF=0.68` defines high-confidence detections and new-track eligibility. Detections from `TRACK_DETECTION_FLOOR=0.50` to below 0.68 are available only for ByteTrack association/recovery. Predictions below 0.50 never enter the tracker.

Tracking continuity is evaluated separately inside each contiguous `VISIBLE` interval. `VISIBLE_TRACK_SEGMENTS_TOTAL` includes the first expected segment in every independent VISIBLE interval; `VISIBLE_EXCESS_FRAGMENTS` counts only segments beyond the first within each interval. Reported `VISIBLE_ID_TRANSITIONS` and `VISIBLE_EXCESS_FRAGMENTS` are diagnostics, not official MOT ID switches or MOT accuracy metrics.

## 1. Experiment metadata and CONFIG

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, importlib.util, json, os, platform, subprocess, sys, time
import cv2
import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml

STARTED_AT = datetime.now(timezone.utc).isoformat()
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks': PROJECT_ROOT = PROJECT_ROOT.parent
CONDA_ENV = os.environ.get('CONDA_DEFAULT_ENV', '')
GIT_COMMIT = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, capture_output=True, text=True, check=True).stdout.strip()
CUDA_AVAILABLE = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else 'NOT_AVAILABLE'

EXPERIMENT_ID = 'FRONT_BYTETRACK_ABLATION_001'
MODEL_PATH = PROJECT_ROOT / 'runs' / 'front' / 'yolov8n_front_v1_baseline' / 'weights' / 'best.pt'
EXPECTED_MODEL_SHA256 = '750b0f8a1621f7214c8122467e5360ada69673ae4a8dc5bf3fd7b1e280287738'
SOURCE_DETECTION_ID = 'FRONT_VIDEO_DET_CONF068_N1_001'
SOURCE_VISIBILITY_ID = 'FRONT_VISIBILITY_AUDIT_CONF068_N1_001'
OPERATIONAL_HIGH_CONF = 0.68
TRACK_DETECTION_FLOOR = 0.50
NMS_IOU = 0.70
IMGSZ = 640
DEVICE = 0
PROGRESS_INTERVAL = 200
FLOAT_TOLERANCE = 1e-6
DETECTION_ONLY_LONGEST_VISIBLE_MISS_FRAMES = 8
TRACKER_VARIANTS = {'B15': 15, 'B30': 30, 'B60': 60}
TRACKER_CONFIG_DIR = PROJECT_ROOT / 'configs' / 'trackers'
TRACKER_CONFIG_PATHS = {name: TRACKER_CONFIG_DIR / f'front_bytetrack_{name.lower()}.yaml' for name in TRACKER_VARIANTS}
SOURCE_DETECTION_CONFIG_PATH = PROJECT_ROOT / 'logs' / 'detection' / SOURCE_DETECTION_ID / 'config.yaml'
SOURCE_DETECTION_SUMMARY_PATH = PROJECT_ROOT / 'logs' / 'detection' / SOURCE_DETECTION_ID / 'summary.json'
SOURCE_VISIBILITY_CONFIG_PATH = PROJECT_ROOT / 'logs' / 'detection' / SOURCE_VISIBILITY_ID / 'config.yaml'
SOURCE_VISIBILITY_SUMMARY_PATH = PROJECT_ROOT / 'logs' / 'detection' / SOURCE_VISIBILITY_ID / 'summary.json'
VISIBILITY_GROUNDTRUTH_PATH = PROJECT_ROOT / 'results' / 'detection' / 'front_visibility_frame_labels.csv'
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'front' / 'tracking'
RESULTS_PATH = PROJECT_ROOT / 'results' / 'tracking' / 'front_bytetrack_ablation.csv'
LOG_DIR = PROJECT_ROOT / 'logs' / 'tracking' / EXPERIMENT_ID

CONFIG = {'experiment_id': EXPERIMENT_ID, 'model_path': str(MODEL_PATH.relative_to(PROJECT_ROOT)), 'expected_model_sha256': EXPECTED_MODEL_SHA256, 'source_detection_id': SOURCE_DETECTION_ID, 'source_visibility_id': SOURCE_VISIBILITY_ID, 'operational_high_conf': OPERATIONAL_HIGH_CONF, 'tracking_detection_floor': TRACK_DETECTION_FLOOR, 'nms_iou': NMS_IOU, 'imgsz': IMGSZ, 'device': DEVICE, 'tracker_variants': TRACKER_VARIANTS, 'tracker_config_paths': {name: str(path.relative_to(PROJECT_ROOT)) for name, path in TRACKER_CONFIG_PATHS.items()}, 'visibility_groundtruth': str(VISIBILITY_GROUNDTRUTH_PATH.relative_to(PROJECT_ROOT)), 'output_root': str(OUTPUT_ROOT.relative_to(PROJECT_ROOT)), 'results_path': str(RESULTS_PATH.relative_to(PROJECT_ROOT))}
print(f'experiment_id: {EXPERIMENT_ID}')
print(f'datetime_utc: {STARTED_AT}')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'Python executable: {sys.executable}')
print(f'Conda environment: {CONDA_ENV}')
print(f'Python: {platform.python_version()}; Torch: {torch.__version__}; Ultralytics: {ultralytics.__version__}')
print(f'Device: {DEVICE}; CUDA available: {CUDA_AVAILABLE}; GPU: {GPU_NAME}')
print(f'Git commit: {GIT_COMMIT}')
print('CONFIG — FRONT BYTETRACK ABLATION')
for key, value in CONFIG.items(): print(f'{key}: {value}')
if importlib.util.find_spec('lap') is None:
    raise ModuleNotFoundError('MISSING_DEPENDENCY: Ultralytics ByteTrack requires lap>=0.5.12. STOP; install the minimum dependency in Conda env fish only after USER approval, then restart kernel and Run All.')

experiment_id: FRONT_BYTETRACK_ABLATION_001
datetime_utc: 2026-08-17T11:48:01.977059+00:00
PROJECT_ROOT: /home/diy-hus/fish
Python executable: /home/diy-hus/miniconda3/envs/fish/bin/python
Conda environment: fish
Python: 3.11.15; Torch: 2.13.0+cu130; Ultralytics: 8.4.120
Device: 0; CUDA available: True; GPU: NVIDIA GeForce RTX 3050
Git commit: de56549b8f773395b1c3fdf0d7841e6559945030
CONFIG — FRONT BYTETRACK ABLATION
experiment_id: FRONT_BYTETRACK_ABLATION_001
model_path: runs/front/yolov8n_front_v1_baseline/weights/best.pt
expected_model_sha256: 750b0f8a1621f7214c8122467e5360ada69673ae4a8dc5bf3fd7b1e280287738
source_detection_id: FRONT_VIDEO_DET_CONF068_N1_001
source_visibility_id: FRONT_VISIBILITY_AUDIT_CONF068_N1_001
operational_high_conf: 0.68
tracking_detection_floor: 0.5
nms_iou: 0.7
imgsz: 640
device: 0
tracker_variants: {'B15': 15, 'B30': 30, 'B60': 60}
tracker_config_paths: {'B15': 'configs/trackers/front_bytetrack_b15.yaml', 'B30': 'configs/trackers/front_bytetrack_b30.yaml',

## 2. Provenance, visibility ground truth, and tracker-config preflight

In [2]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''): digest.update(chunk)
    return digest.hexdigest()

assert CONDA_ENV == 'fish', f'FAIL preflight: expected Conda env fish, found {CONDA_ENV!r}'
assert CUDA_AVAILABLE, 'FAIL preflight: CUDA is required because DEVICE=0.'
assert 0.0 <= TRACK_DETECTION_FLOOR <= OPERATIONAL_HIGH_CONF <= 1.0
assert MODEL_PATH.is_file(), f'FAIL preflight: missing model {MODEL_PATH}'
MODEL_SHA256 = sha256_file(MODEL_PATH)
assert MODEL_SHA256 == EXPECTED_MODEL_SHA256, f'FAIL provenance: model SHA-256 mismatch: {MODEL_SHA256}'
for required in (SOURCE_DETECTION_CONFIG_PATH, SOURCE_DETECTION_SUMMARY_PATH, SOURCE_VISIBILITY_CONFIG_PATH, SOURCE_VISIBILITY_SUMMARY_PATH, VISIBILITY_GROUNDTRUTH_PATH):
    assert required.is_file(), f'FAIL preflight: missing evidence {required.relative_to(PROJECT_ROOT)}'
SOURCE_DETECTION_CONFIG = yaml.safe_load(SOURCE_DETECTION_CONFIG_PATH.read_text(encoding='utf-8'))
SOURCE_DETECTION_SUMMARY = json.loads(SOURCE_DETECTION_SUMMARY_PATH.read_text(encoding='utf-8'))
SOURCE_VISIBILITY_CONFIG = yaml.safe_load(SOURCE_VISIBILITY_CONFIG_PATH.read_text(encoding='utf-8'))
SOURCE_VISIBILITY_SUMMARY = json.loads(SOURCE_VISIBILITY_SUMMARY_PATH.read_text(encoding='utf-8'))
assert SOURCE_DETECTION_CONFIG['experiment_id'] == SOURCE_DETECTION_ID
assert SOURCE_VISIBILITY_CONFIG['experiment_id'] == SOURCE_VISIBILITY_ID
assert SOURCE_DETECTION_SUMMARY['model_sha256'] == SOURCE_VISIBILITY_SUMMARY['model_sha256'] == MODEL_SHA256
assert np.isclose(float(SOURCE_DETECTION_CONFIG['detection_conf']), OPERATIONAL_HIGH_CONF), 'FAIL provenance: operational confidence differs from Notebook 05.'
assert np.isclose(float(SOURCE_DETECTION_CONFIG['nms_iou']), NMS_IOU), 'FAIL provenance: NMS IoU differs from Notebook 05.'
assert int(SOURCE_DETECTION_CONFIG['imgsz']) == IMGSZ, 'FAIL provenance: image size differs from Notebook 05.'
VIDEO_PATH = PROJECT_ROOT / SOURCE_DETECTION_CONFIG['video_path']
assert VIDEO_PATH.is_file(), f'FAIL preflight: source video missing: {VIDEO_PATH}'
VIDEO_SHA256 = sha256_file(VIDEO_PATH)
assert VIDEO_SHA256 == SOURCE_DETECTION_CONFIG['video_sha256'] == SOURCE_DETECTION_SUMMARY['video_sha256'] == SOURCE_VISIBILITY_SUMMARY['video_sha256'], 'FAIL provenance: video SHA-256 mismatch.'
VIDEO_FPS = float(SOURCE_DETECTION_CONFIG['video_fps'])
TOTAL_FRAMES = int(SOURCE_DETECTION_CONFIG['video_frame_count'])
VIDEO_WIDTH, VIDEO_HEIGHT = map(int, SOURCE_DETECTION_CONFIG['video_resolution'].split('x'))
VISIBILITY_LABELS = pd.read_csv(VISIBILITY_GROUNDTRUTH_PATH).sort_values('frame_index').reset_index(drop=True)
required_visibility_columns = {'frame_index', 'time_sec', 'visibility_state', 'expected_visible_count', 'visibility_source'}
assert required_visibility_columns.issubset(VISIBILITY_LABELS.columns), f'FAIL visibility schema: missing {required_visibility_columns - set(VISIBILITY_LABELS.columns)}'
assert len(VISIBILITY_LABELS) == TOTAL_FRAMES and VISIBILITY_LABELS['frame_index'].is_unique
assert np.array_equal(VISIBILITY_LABELS['frame_index'].to_numpy(), np.arange(TOTAL_FRAMES)), 'FAIL visibility coverage: frame indices are not contiguous.'
assert set(VISIBILITY_LABELS['visibility_state']).issubset({'VISIBLE', 'IN_SHELTER', 'PARTIALLY_VISIBLE', 'UNCERTAIN'})
assert int(SOURCE_VISIBILITY_SUMMARY['longest_visible_miss_run_frames']) == DETECTION_ONLY_LONGEST_VISIBLE_MISS_FRAMES
VISIBILITY_LABELS['visible_interval_id'] = pd.Series(pd.NA, index=VISIBILITY_LABELS.index, dtype='Int64')
visible_interval_id = 0; previous_visible = False
for row_index, state in enumerate(VISIBILITY_LABELS['visibility_state']):
    is_visible = state == 'VISIBLE'
    if is_visible and not previous_visible: visible_interval_id += 1
    if is_visible: VISIBILITY_LABELS.at[row_index, 'visible_interval_id'] = visible_interval_id
    previous_visible = is_visible
VISIBLE_FRAMES = int((VISIBILITY_LABELS['visibility_state'] == 'VISIBLE').sum())
assert VISIBLE_FRAMES > 0 and visible_interval_id > 0

EXPECTED_TRACKER_BASE = {'tracker_type': 'bytetrack', 'track_high_thresh': OPERATIONAL_HIGH_CONF, 'track_low_thresh': TRACK_DETECTION_FLOOR, 'new_track_thresh': OPERATIONAL_HIGH_CONF, 'match_thresh': 0.80, 'fuse_score': True}
TRACKER_CONFIGS = {}
for variant, expected_buffer in TRACKER_VARIANTS.items():
    path = TRACKER_CONFIG_PATHS[variant]
    assert path.is_file(), f'FAIL tracker config: missing {path.relative_to(PROJECT_ROOT)}'
    tracker_config = yaml.safe_load(path.read_text(encoding='utf-8'))
    assert tracker_config['tracker_type'] == EXPECTED_TRACKER_BASE['tracker_type']
    for field in ('track_high_thresh', 'track_low_thresh', 'new_track_thresh', 'match_thresh'):
        assert np.isclose(float(tracker_config[field]), float(EXPECTED_TRACKER_BASE[field])), f'FAIL tracker config {variant}: {field}'
    assert bool(tracker_config['fuse_score']) is True
    assert int(tracker_config['track_buffer']) == expected_buffer
    assert 0.0 <= float(tracker_config['track_low_thresh']) <= float(tracker_config['track_high_thresh']) <= 1.0
    assert float(tracker_config['track_low_thresh']) >= TRACK_DETECTION_FLOOR
    assert float(tracker_config['new_track_thresh']) >= OPERATIONAL_HIGH_CONF
    TRACKER_CONFIGS[variant] = tracker_config
for output_dir in (OUTPUT_ROOT / name for name in TRACKER_VARIANTS):
    if output_dir.exists() and any(output_dir.iterdir()): raise RuntimeError(f'FAIL preflight: preserve existing output before rerun: {output_dir.relative_to(PROJECT_ROOT)}')
if RESULTS_PATH.exists() or (LOG_DIR.exists() and any(LOG_DIR.iterdir())): raise RuntimeError('FAIL preflight: preserve existing Checkpoint 07 evidence before rerun.')
print(f'Video: {VIDEO_PATH.relative_to(PROJECT_ROOT)}; SHA-256={VIDEO_SHA256}; frames={TOTAL_FRAMES}; FPS={VIDEO_FPS:.6f}; resolution={VIDEO_WIDTH}x{VIDEO_HEIGHT}')
print(f'Visibility ground truth: loaded; visible frames={VISIBLE_FRAMES}; contiguous visible intervals={visible_interval_id}')
print('TRACKER CONFIG PREFLIGHT: PASS')
for name, config in TRACKER_CONFIGS.items(): print(name, config)

Video: data/raw/front/4.mp4; SHA-256=3f8587344beba8dcc6f835d1ed94aa8daadb06a369b5b96977ee902f035b5700; frames=3431; FPS=28.668432; resolution=1280x960
Visibility ground truth: loaded; visible frames=2335; contiguous visible intervals=18
TRACKER CONFIG PREFLIGHT: PASS
B15 {'tracker_type': 'bytetrack', 'track_high_thresh': 0.68, 'track_low_thresh': 0.5, 'new_track_thresh': 0.68, 'track_buffer': 15, 'match_thresh': 0.8, 'fuse_score': True}
B30 {'tracker_type': 'bytetrack', 'track_high_thresh': 0.68, 'track_low_thresh': 0.5, 'new_track_thresh': 0.68, 'track_buffer': 30, 'match_thresh': 0.8, 'fuse_score': True}
B60 {'tracker_type': 'bytetrack', 'track_high_thresh': 0.68, 'track_low_thresh': 0.5, 'new_track_thresh': 0.68, 'track_buffer': 60, 'match_thresh': 0.8, 'fuse_score': True}


## 3. Run the fixed ByteTrack ablation

Only `track_buffer` changes. A fresh YOLO/ByteTrack state is created for every configuration. Rows represent observed tracked boxes returned for the current frame; internal lost tracks are not emitted as observed fish boxes. A single NaN `track_id` row is written when no active tracked bbox exists.

In [3]:
from ultralytics import YOLO

TRACKING_TABLES = {}
PROCESSING_STATS = {}
ACTUAL_TRACKER_CONFIGS = {}
for variant, track_buffer in TRACKER_VARIANTS.items():
    tracker_path = TRACKER_CONFIG_PATHS[variant]
    output_dir = OUTPUT_ROOT / variant
    output_dir.mkdir(parents=True, exist_ok=False)
    overlay_path = output_dir / 'front_bytetrack_overlay.mp4'
    tracks_path = output_dir / 'frame_tracks.csv'
    model = YOLO(str(MODEL_PATH), task='detect')
    capture = cv2.VideoCapture(str(VIDEO_PATH))
    if not capture.isOpened(): raise RuntimeError(f'FAIL {variant}: cannot open video.')
    writer = cv2.VideoWriter(str(overlay_path), cv2.VideoWriter_fourcc(*'mp4v'), VIDEO_FPS, (VIDEO_WIDTH, VIDEO_HEIGHT))
    if not writer.isOpened(): capture.release(); raise RuntimeError(f'FAIL {variant}: cannot create overlay.')
    rows = []; frame_index = 0; run_start = time.perf_counter(); actual_config_checked = False
    print(f'{variant} START — track_buffer={track_buffer}; detector_floor={TRACK_DETECTION_FLOOR}; high/new={OPERATIONAL_HIGH_CONF}')
    try:
        while True:
            ok, frame = capture.read()
            if not ok: break
            visibility = VISIBILITY_LABELS.iloc[frame_index]['visibility_state']
            time_sec = frame_index / VIDEO_FPS
            result = model.track(source=frame, persist=True, tracker=str(tracker_path), conf=TRACK_DETECTION_FLOOR, iou=NMS_IOU, imgsz=IMGSZ, device=DEVICE, verbose=False)[0]
            if not actual_config_checked:
                tracker = model.predictor.trackers[0]
                actual = {field: getattr(tracker.args, field) for field in ('tracker_type', 'track_high_thresh', 'track_low_thresh', 'new_track_thresh', 'track_buffer', 'match_thresh', 'fuse_score')}
                expected = TRACKER_CONFIGS[variant]
                assert actual['tracker_type'] == 'bytetrack'
                for field in ('track_high_thresh', 'track_low_thresh', 'new_track_thresh', 'match_thresh'):
                    assert np.isclose(float(actual[field]), float(expected[field])), f'FAIL {variant}: actual tracker {field}={actual[field]}'
                assert int(actual['track_buffer']) == track_buffer and bool(actual['fuse_score']) is True
                assert float(actual['track_low_thresh']) >= TRACK_DETECTION_FLOOR
                ACTUAL_TRACKER_CONFIGS[variant] = actual
                print(f'{variant} ACTUAL TRACKER CONFIG: {actual}')
                actual_config_checked = True
            boxes = result.boxes
            observed = boxes is not None and boxes.id is not None and len(boxes.id) > 0
            overlay = frame.copy()
            if observed:
                track_ids = boxes.id.detach().cpu().numpy().astype(int)
                confidences = boxes.conf.detach().cpu().numpy().astype(float)
                xyxy = boxes.xyxy.detach().cpu().numpy().astype(float)
                assert float(confidences.min()) >= TRACK_DETECTION_FLOOR - FLOAT_TOLERANCE, f'FAIL {variant}: detection below tracker floor entered output.'
                for track_id, confidence, coords in zip(track_ids, confidences, xyxy):
                    x1, y1, x2, y2 = coords.tolist(); cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
                    rows.append({'frame_index': frame_index, 'time_sec': time_sec, 'visibility_state': visibility, 'track_id': int(track_id), 'confidence': float(confidence), 'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2, 'cx': cx, 'cy': cy})
                    p1, p2 = (int(round(x1)), int(round(y1))), (int(round(x2)), int(round(y2)))
                    cv2.rectangle(overlay, p1, p2, (0, 220, 0), 2)
                    cv2.putText(overlay, f'ID {track_id} | {confidence:.2f}', (p1[0], max(18, p1[1] - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 220, 0), 2, cv2.LINE_AA)
            else:
                rows.append({'frame_index': frame_index, 'time_sec': time_sec, 'visibility_state': visibility, 'track_id': np.nan, 'confidence': np.nan, 'x1': np.nan, 'y1': np.nan, 'x2': np.nan, 'y2': np.nan, 'cx': np.nan, 'cy': np.nan})
            active_count = int(boxes.id.numel()) if observed else 0
            header = f'{variant} | frame={frame_index}/{TOTAL_FRAMES - 1} | t={time_sec:.2f}s | state={visibility} | active_tracks={active_count}'
            cv2.rectangle(overlay, (0, 0), (min(VIDEO_WIDTH, 1000), 36), (0, 0, 0), -1)
            cv2.putText(overlay, header, (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.64, (255, 255, 255), 2, cv2.LINE_AA)
            writer.write(overlay); frame_index += 1
            if frame_index % PROGRESS_INTERVAL == 0 or frame_index == TOTAL_FRAMES:
                elapsed = time.perf_counter() - run_start
                print(f'{variant}: {frame_index}/{TOTAL_FRAMES} | elapsed={elapsed:.1f}s | processing FPS={frame_index / elapsed:.2f}')
    finally:
        capture.release(); writer.release()
    runtime_sec = time.perf_counter() - run_start
    assert frame_index == TOTAL_FRAMES, f'FAIL {variant}: incomplete inference {frame_index}/{TOTAL_FRAMES}'
    assert actual_config_checked, f'FAIL {variant}: tracker configuration was not inspected.'
    tracking_df = pd.DataFrame(rows, columns=['frame_index', 'time_sec', 'visibility_state', 'track_id', 'confidence', 'x1', 'y1', 'x2', 'y2', 'cx', 'cy'])
    assert tracking_df['frame_index'].nunique() == TOTAL_FRAMES
    tracking_df.to_csv(tracks_path, index=False)
    TRACKING_TABLES[variant] = tracking_df
    PROCESSING_STATS[variant] = {'runtime_sec': runtime_sec, 'processing_FPS': TOTAL_FRAMES / runtime_sec, 'tracks_path': str(tracks_path.relative_to(PROJECT_ROOT)), 'overlay_path': str(overlay_path.relative_to(PROJECT_ROOT))}
    print(f'{variant} COMPLETE — rows={len(tracking_df)}; processing FPS={TOTAL_FRAMES / runtime_sec:.2f}; outputs={output_dir.relative_to(PROJECT_ROOT)}')

B15 START — track_buffer=15; detector_floor=0.5; high/new=0.68
B15 ACTUAL TRACKER CONFIG: {'tracker_type': 'bytetrack', 'track_high_thresh': 0.68, 'track_low_thresh': 0.5, 'new_track_thresh': 0.68, 'track_buffer': 15, 'match_thresh': 0.8, 'fuse_score': True}
B15: 200/3431 | elapsed=6.0s | processing FPS=33.32
B15: 400/3431 | elapsed=9.7s | processing FPS=41.20
B15: 600/3431 | elapsed=13.4s | processing FPS=44.75
B15: 800/3431 | elapsed=17.3s | processing FPS=46.37
B15: 1000/3431 | elapsed=21.2s | processing FPS=47.27
B15: 1200/3431 | elapsed=24.5s | processing FPS=49.06
B15: 1400/3431 | elapsed=27.7s | processing FPS=50.50
B15: 1600/3431 | elapsed=31.0s | processing FPS=51.58
B15: 1800/3431 | elapsed=34.4s | processing FPS=52.37
B15: 2000/3431 | elapsed=37.7s | processing FPS=53.05
B15: 2200/3431 | elapsed=41.5s | processing FPS=53.08
B15: 2400/3431 | elapsed=45.3s | processing FPS=53.03
B15: 2600/3431 | elapsed=49.2s | processing FPS=52.86
B15: 2800/3431 | elapsed=53.0s | processing F

## 4. Visibility-aware tracking diagnostics

A visible frame is tracked when at least one observed track box exists. For single-fish continuity, a fragment is a consecutive run of frames with exactly one observed track and the same ID. Zero-track or multi-track frames break fragments. `VISIBLE_ID_TRANSITIONS` counts changes between successive single-track observations within the same contiguous VISIBLE interval, even when separated by an untracked gap; it never crosses a non-VISIBLE state.

In [4]:
def run_lengths(frame_indices):
    values = np.asarray(frame_indices, dtype=int)
    if not len(values): return []
    lengths = []; start = previous = int(values[0])
    for value in values[1:]:
        value = int(value)
        if value != previous + 1:
            lengths.append(previous - start + 1); start = value
        previous = value
    lengths.append(previous - start + 1)
    return lengths

COMPARISON_ROWS = []; INTERVAL_DIAGNOSTICS = {}; LIFESPAN_DIAGNOSTICS = {}
for variant, tracking_df in TRACKING_TABLES.items():
    observed = tracking_df[tracking_df['track_id'].notna()].copy()
    observed['track_id'] = observed['track_id'].astype(int)
    frame_active_counts = observed.groupby('frame_index').size().reindex(np.arange(TOTAL_FRAMES), fill_value=0).astype(int)
    frame_status = VISIBILITY_LABELS[['frame_index', 'time_sec', 'visibility_state', 'visible_interval_id']].copy()
    frame_status['active_track_count'] = frame_active_counts.to_numpy()
    visible_status = frame_status[frame_status['visibility_state'] == 'VISIBLE'].copy()
    visible_tracked_frames = int((visible_status['active_track_count'] >= 1).sum())
    visible_untracked_frames = int((visible_status['active_track_count'] == 0).sum())
    visible_multi_track_frames = int((visible_status['active_track_count'] > 1).sum())
    visible_one_track_frames = int((visible_status['active_track_count'] == 1).sum())
    interval_rows = []; total_transitions = 0; total_track_segments = 0; total_excess_fragments = 0; segment_lengths = []; all_untracked_lengths = []
    for interval_id, interval_frames in visible_status.groupby('visible_interval_id', sort=True):
        interval_frame_ids = interval_frames['frame_index'].to_numpy(dtype=int)
        interval_observed = observed[observed['frame_index'].isin(interval_frame_ids)]
        unique_ids = int(interval_observed['track_id'].nunique())
        last_observed_id = None; active_fragment_id = None; current_fragment_length = 0; transitions = 0; fragments = 0; local_fragment_lengths = []
        for frame_index in interval_frame_ids:
            ids = interval_observed.loc[interval_observed['frame_index'] == frame_index, 'track_id'].tolist()
            if len(ids) == 1:
                current_id = int(ids[0])
                if active_fragment_id != current_id:
                    if current_fragment_length: local_fragment_lengths.append(current_fragment_length)
                    fragments += 1; current_fragment_length = 0; active_fragment_id = current_id
                if last_observed_id is not None and current_id != last_observed_id: transitions += 1
                last_observed_id = current_id; current_fragment_length += 1
            else:
                if current_fragment_length: local_fragment_lengths.append(current_fragment_length)
                current_fragment_length = 0; active_fragment_id = None
        if current_fragment_length: local_fragment_lengths.append(current_fragment_length)
        untracked_lengths = run_lengths(interval_frames.loc[interval_frames['active_track_count'] == 0, 'frame_index'])
        excess_fragments = max(0, fragments - 1)
        total_transitions += transitions; total_track_segments += fragments; total_excess_fragments += excess_fragments; segment_lengths.extend(local_fragment_lengths); all_untracked_lengths.extend(untracked_lengths)
        interval_rows.append({'visible_interval_id': int(interval_id), 'start_frame': int(interval_frame_ids.min()), 'end_frame': int(interval_frame_ids.max()), 'visible_frames': len(interval_frame_ids), 'unique_track_ids_visible': unique_ids, 'visible_id_transitions': transitions, 'track_segments_in_interval': fragments, 'excess_track_fragments': excess_fragments, 'longest_untracked_run_frames': max(untracked_lengths, default=0), 'mean_track_segment_duration_frames': float(np.mean(local_fragment_lengths)) if local_fragment_lengths else 0.0, 'mean_track_segment_duration_sec': float(np.mean(local_fragment_lengths) / VIDEO_FPS) if local_fragment_lengths else 0.0})
    interval_df = pd.DataFrame(interval_rows)
    INTERVAL_DIAGNOSTICS[variant] = interval_df
    lifespan_df = observed.groupby('track_id').agg(first_frame=('frame_index', 'min'), last_frame=('frame_index', 'max'), observed_frames=('frame_index', 'nunique')).reset_index()
    lifespan_df['lifespan_sec'] = lifespan_df['observed_frames'] / VIDEO_FPS
    LIFESPAN_DIAGNOSTICS[variant] = lifespan_df
    shelter_boxes = int((observed['visibility_state'] == 'IN_SHELTER').sum())
    longest_untracked = max(all_untracked_lengths, default=0)
    comparison_row = {'configuration': variant, 'track_buffer': TRACKER_VARIANTS[variant], 'visible_interval_count': int(visible_status['visible_interval_id'].nunique()), 'visible_frames': VISIBLE_FRAMES, 'visible_tracked_frames': visible_tracked_frames, 'visible_untracked_frames': visible_untracked_frames, 'visible_track_coverage': visible_tracked_frames / VISIBLE_FRAMES, 'visible_zero_track_rate': visible_untracked_frames / VISIBLE_FRAMES, 'visible_one_track_rate': visible_one_track_frames / VISIBLE_FRAMES, 'visible_multi_track_rate': visible_multi_track_frames / VISIBLE_FRAMES, 'visible_id_transitions': total_transitions, 'visible_track_segments_total': total_track_segments, 'visible_excess_fragments': total_excess_fragments, 'untracked_runs': len(all_untracked_lengths), 'longest_visible_untracked_run_frames': longest_untracked, 'longest_visible_untracked_run_sec': longest_untracked / VIDEO_FPS, 'median_visible_untracked_run_frames': float(np.median(all_untracked_lengths)) if all_untracked_lengths else 0.0, 'p95_visible_untracked_run_frames': float(np.quantile(all_untracked_lengths, 0.95)) if all_untracked_lengths else 0.0, 'mean_track_segment_duration_sec': float(np.mean(segment_lengths) / VIDEO_FPS) if segment_lengths else 0.0, 'observed_track_boxes_during_shelter': shelter_boxes, 'unique_track_ids_total': int(observed['track_id'].nunique()), 'median_track_lifespan_sec': float(lifespan_df['lifespan_sec'].median()) if len(lifespan_df) else 0.0, 'mean_track_lifespan_sec': float(lifespan_df['lifespan_sec'].mean()) if len(lifespan_df) else 0.0, 'max_track_lifespan_sec': float(lifespan_df['lifespan_sec'].max()) if len(lifespan_df) else 0.0, 'processing_FPS': PROCESSING_STATS[variant]['processing_FPS']}
    COMPARISON_ROWS.append(comparison_row)
COMPARISON_DF = pd.DataFrame(COMPARISON_ROWS).sort_values('track_buffer').reset_index(drop=True)
display(COMPARISON_DF)
print(f'Detection-only longest VISIBLE miss baseline: {DETECTION_ONLY_LONGEST_VISIBLE_MISS_FRAMES} frames.')
print('These are visibility-aware single-fish tracking diagnostics, not official MOT metrics.')

,configuration,track_buffer,visible_frames,visible_tracked_frames,visible_untracked_frames,visible_track_coverage,visible_zero_track_rate,visible_one_track_rate,visible_multi_track_rate,visible_id_transitions,...,longest_visible_untracked_run_sec,median_visible_untracked_run_frames,p95_visible_untracked_run_frames,mean_track_fragment_duration_sec,observed_track_boxes_during_shelter,unique_track_ids_total,median_track_lifespan_sec,mean_track_lifespan_sec,max_track_lifespan_sec,processing_FPS
0,B15,15,2335,2333,2,0.999143,0.000857,0.999143,0.0,0,...,0.034882,1.0,1.0,4.28309,6,2,41.509072,41.509072,49.148136,52.409485
1,B30,30,2335,2333,2,0.999143,0.000857,0.999143,0.0,0,...,0.034882,1.0,1.0,4.28309,6,2,41.509072,41.509072,49.148136,50.404808
2,B60,60,2335,2333,2,0.999143,0.000857,0.999143,0.0,0,...,0.034882,1.0,1.0,4.28309,6,2,41.509072,41.509072,49.148136,52.373333


Detection-only longest VISIBLE miss baseline: 8 frames.
These are visibility-aware single-fish tracking diagnostics, not official MOT metrics.


## 5. Candidate ranking and evidence

In [5]:
ranking = COMPARISON_DF.sort_values(['visible_track_coverage', 'visible_id_transitions', 'visible_excess_fragments', 'longest_visible_untracked_run_frames', 'visible_multi_track_rate', 'track_buffer'], ascending=[False, True, True, True, True, True]).reset_index(drop=True)
BEST_CANDIDATE = str(ranking.iloc[0]['configuration'])
SELECTION_REASON = 'B15 provisional candidate: B15/B30/B60 have equivalent visible coverage, within-interval ID transitions, excess fragments, visible gaps, and duplicate-track rate; B15 has the smallest buffer on the quality-metric tie. USER makes the final selection.'
WARNINGS = []
for row in COMPARISON_DF.itertuples(index=False):
    if row.visible_track_coverage < 1.0 or row.visible_id_transitions > 0 or row.visible_excess_fragments > 0 or row.visible_multi_track_rate > 0 or row.observed_track_boxes_during_shelter > 0:
        WARNINGS.append(f'{row.configuration} has imperfect visibility-aware tracking diagnostics; inspect metrics and overlay.')
CHECKPOINT_RESULT = 'PASS_WITH_WARNING' if WARNINGS else 'PASS'
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=False)
COMPARISON_DF.to_csv(RESULTS_PATH, index=False)
for variant in TRACKER_VARIANTS:
    INTERVAL_DIAGNOSTICS[variant].to_csv(OUTPUT_ROOT / variant / 'visible_interval_diagnostics.csv', index=False)
    LIFESPAN_DIAGNOSTICS[variant].to_csv(OUTPUT_ROOT / variant / 'track_lifespans.csv', index=False)
CONFIG_PATH = LOG_DIR / 'config.yaml'; ENVIRONMENT_PATH = LOG_DIR / 'environment.txt'; SUMMARY_PATH = LOG_DIR / 'summary.json'
CONFIG_EVIDENCE = {**CONFIG, 'model_sha256': MODEL_SHA256, 'video': str(VIDEO_PATH.relative_to(PROJECT_ROOT)), 'video_sha256': VIDEO_SHA256, 'video_fps': VIDEO_FPS, 'total_frames': TOTAL_FRAMES, 'visibility_groundtruth_sha256': sha256_file(VISIBILITY_GROUNDTRUTH_PATH), 'requested_tracker_configs': TRACKER_CONFIGS, 'actual_tracker_configs': ACTUAL_TRACKER_CONFIGS, 'git_commit': GIT_COMMIT}
CONFIG_PATH.write_text(yaml.safe_dump(CONFIG_EVIDENCE, sort_keys=False), encoding='utf-8')
ENVIRONMENT_LINES = [f'experiment_id={EXPERIMENT_ID}', f'datetime_utc={STARTED_AT}', f'git_commit={GIT_COMMIT}', f'python_executable={sys.executable}', f'python={platform.python_version()}', f'conda_env={CONDA_ENV}', f'torch={torch.__version__}', f'cuda_runtime={torch.version.cuda}', f'cuda_available={CUDA_AVAILABLE}', f'gpu={GPU_NAME}', f'ultralytics={ultralytics.__version__}', f'opencv={cv2.__version__}']
ENVIRONMENT_PATH.write_text('\n'.join(ENVIRONMENT_LINES) + '\n', encoding='utf-8')
OUTPUT_FILES = [RESULTS_PATH, CONFIG_PATH, ENVIRONMENT_PATH, SUMMARY_PATH] + [OUTPUT_ROOT / variant / filename for variant in TRACKER_VARIANTS for filename in ('frame_tracks.csv', 'front_bytetrack_overlay.mp4', 'visible_interval_diagnostics.csv', 'track_lifespans.csv')]
SUMMARY = {'experiment_id': EXPERIMENT_ID, 'model_sha256': MODEL_SHA256, 'video_sha256': VIDEO_SHA256, 'operational_high_conf': OPERATIONAL_HIGH_CONF, 'tracking_detection_floor': TRACK_DETECTION_FLOOR, 'visibility_groundtruth': 'loaded', 'detection_only_longest_visible_miss_frames': DETECTION_ONLY_LONGEST_VISIBLE_MISS_FRAMES, 'configurations': {row.configuration: row._asdict() for row in COMPARISON_DF.itertuples(index=False)}, 'actual_tracker_configs': ACTUAL_TRACKER_CONFIGS, 'best_candidate': BEST_CANDIDATE, 'selection_reason': SELECTION_REASON, 'checkpoint_result': CHECKPOINT_RESULT, 'warnings': WARNINGS, 'output_files': [str(path.relative_to(PROJECT_ROOT)) for path in OUTPUT_FILES], 'next_step': 'USER reviews ByteTrack ablation before Notebook 08.'}
SUMMARY_PATH.write_text(json.dumps(SUMMARY, indent=2, ensure_ascii=False), encoding='utf-8')
for path in OUTPUT_FILES: print(f'Created {path.relative_to(PROJECT_ROOT)} ({path.stat().st_size} bytes)')

Created results/tracking/front_bytetrack_ablation.csv (1210 bytes)
Created logs/tracking/FRONT_BYTETRACK_ABLATION_001/config.yaml (2304 bytes)
Created logs/tracking/FRONT_BYTETRACK_ABLATION_001/environment.txt (354 bytes)
Created logs/tracking/FRONT_BYTETRACK_ABLATION_001/summary.json (5911 bytes)
Created outputs/front/tracking/B15/frame_tracks.csv (427496 bytes)
Created outputs/front/tracking/B15/front_bytetrack_overlay.mp4 (75391334 bytes)
Created outputs/front/tracking/B15/visible_interval_diagnostics.csv (1065 bytes)
Created outputs/front/tracking/B15/track_lifespans.csv (126 bytes)
Created outputs/front/tracking/B30/frame_tracks.csv (427496 bytes)
Created outputs/front/tracking/B30/front_bytetrack_overlay.mp4 (75400333 bytes)
Created outputs/front/tracking/B30/visible_interval_diagnostics.csv (1065 bytes)
Created outputs/front/tracking/B30/track_lifespans.csv (126 bytes)
Created outputs/front/tracking/B60/frame_tracks.csv (427496 bytes)
Created outputs/front/tracking/B60/front_byt

## 6. Final Summary

In [6]:
FINAL_SUMMARY = {'experiment_id': EXPERIMENT_ID, 'model_sha256': MODEL_SHA256, 'video_sha256': VIDEO_SHA256, 'operational_high_conf': OPERATIONAL_HIGH_CONF, 'tracking_detection_floor': TRACK_DETECTION_FLOOR, 'visibility_groundtruth': 'loaded'}
for variant in TRACKER_VARIANTS:
    row = COMPARISON_DF.set_index('configuration').loc[variant]
    FINAL_SUMMARY[variant] = {'visible_track_coverage': row.visible_track_coverage, 'visible_id_transitions': int(row.visible_id_transitions), 'visible_track_segments_total': int(row.visible_track_segments_total), 'visible_excess_fragments': int(row.visible_excess_fragments), 'longest_visible_untracked_run_sec': row.longest_visible_untracked_run_sec, 'unique_track_ids_total': int(row.unique_track_ids_total), 'processing_FPS': row.processing_FPS}
FINAL_SUMMARY.update({'best_candidate': BEST_CANDIDATE, 'selection_reason': SELECTION_REASON, 'checkpoint_result': CHECKPOINT_RESULT, 'warnings': WARNINGS, 'output_files': [str(path.relative_to(PROJECT_ROOT)) for path in OUTPUT_FILES], 'next_step': 'USER reviews ByteTrack ablation before Notebook 08.'})
print('FINAL SUMMARY')
for key, value in FINAL_SUMMARY.items(): print(f'{key}: {value}')

FINAL SUMMARY
experiment_id: FRONT_BYTETRACK_ABLATION_001
model_sha256: 750b0f8a1621f7214c8122467e5360ada69673ae4a8dc5bf3fd7b1e280287738
video_sha256: 3f8587344beba8dcc6f835d1ed94aa8daadb06a369b5b96977ee902f035b5700
operational_high_conf: 0.68
tracking_detection_floor: 0.5
visibility_groundtruth: loaded
B15: {'visible_track_coverage': np.float64(0.9991434689507495), 'visible_id_transitions': 0, 'visible_track_segments_total': 19, 'visible_excess_fragments': 1, 'longest_visible_untracked_run_sec': np.float64(0.03488157291363062), 'unique_track_ids_total': 2, 'processing_FPS': np.float64(52.40948494789904)}
B30: {'visible_track_coverage': np.float64(0.9991434689507495), 'visible_id_transitions': 0, 'visible_track_segments_total': 19, 'visible_excess_fragments': 1, 'longest_visible_untracked_run_sec': np.float64(0.03488157291363062), 'unique_track_ids_total': 2, 'processing_FPS': np.float64(50.404808319238114)}
B60: {'visible_track_coverage': np.float64(0.9991434689507495), 'visible_id_tr